In [1]:
import os
# ⚠️ MUST be set BEFORE torch is imported — use GPU 1 (GPU 0 is full)
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

print("🔍 Checking environment...")
try:
    import torch, transformers, trl, peft, bitsandbytes, datasets
    print(f"✅ All packages present  (torch {torch.__version__}, transformers {transformers.__version__})")
    if torch.cuda.is_available():
        print(f"🎯 Using GPU: {torch.cuda.get_device_name(0)}")
        print(f"   Free VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3:.1f} GB")
except ImportError as e:
    print(f"⚠️ Missing: {e}\n📦 Installing…")
    %pip install -q torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
    %pip install -q accelerate datasets pyarrow scipy bitsandbytes peft "transformers>=4.46.0" "trl>=0.11.0"
    print("✅ Done — restart kernel, then re-run this cell.")

🔍 Checking environment...
✅ All packages present  (torch 2.5.1+cu121, transformers 5.2.0)
🎯 Using GPU: NVIDIA GeForce RTX 3090
   Free VRAM: 23.6 GB


In [2]:
# 🔎 GPU Diagnostic — run this FIRST to see what's using the GPU
import subprocess, os

print("="*60)
print("GPU STATUS (nvidia-smi)")
print("="*60)
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

GPU STATUS (nvidia-smi)
Mon Feb 23 00:05:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
| 81%   73C    P2            345W /  350W |   23489MiB /  24576MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------

In [3]:
import sys, os

# Debug: show where the kernel thinks we are
print(f"CWD: {os.getcwd()}")
print(f"dorado/ exists here? {os.path.isdir('dorado')}")

# Try common locations to find the dorado package
candidates = [
    os.path.abspath("."),                       # current dir
    os.path.expanduser("~/llmpt"),              # home/llmpt
    "/workspaces/llmpt",                        # devcontainer
]

found = False
for p in candidates:
    if os.path.isdir(os.path.join(p, "dorado")):
        if p not in sys.path:
            sys.path.insert(0, p)
        os.chdir(p)  # also set CWD so relative paths work later
        print(f"✅ Found dorado at: {p}")
        found = True
        break

if not found:
    print("❌ Could not find dorado/ — add your repo path to 'candidates' above")
    print(f"   sys.path = {sys.path[:5]}")

CWD: /home/jovyan
dorado/ exists here? False
✅ Found dorado at: /home/jovyan/llmpt


In [4]:
# 1) Import the dorado package (auto-reloads on code changes)
%load_ext autoreload
%autoreload 2

from dorado import (
    # config / grid
    build_experiment_grid, estimate_time, make_results_paths,
    DATASET_CONFIG, MODEL_CONFIG, ARCHITECTURE_CONFIG,
    TRAINING_CONFIG, GENERATION_CONFIG, EVAL_CONFIG,
    DUAL_PREFERENCE_CONFIG,
    # stages (can call individually for debugging)
    run_sft_stage, run_candidate_generation, run_labeling_stage,
    run_rm_training, run_dpo_training, run_full_evaluation,
    # orchestrator
    run_single_experiment, run_all_experiments, cleanup_artifacts,
    # utils
    clear_gpu, set_random_seeds, cleanup_storage,
)

print("✅ dorado package loaded")

✅ dorado package loaded


In [5]:
EXPERIMENTS = build_experiment_grid()
_, RESULTS_FILE, CHECKPOINT_FILE = make_results_paths()

print(f"Experiments: {len(EXPERIMENTS)}")
total_min = sum(estimate_time(e)["total_min"] for e in EXPERIMENTS)
print(f"Est. total:  {total_min:.1f} min ({total_min/60:.1f} h)")
print(f"Results:     {RESULTS_FILE}")

Experiments: 2
Est. total:  0.6 min (0.0 h)
Results:     results/2026-02-23/dorado_results.xlsx


### 🚀 Run Experiments

The cell below runs every experiment in the grid, checkpointing after each one. Results are exported to Excel at the end.

**To resume** after a crash/restart: just re-run — completed experiments are skipped automatically via the checkpoint file.


In [6]:
results_df = run_all_experiments(EXPERIMENTS, RESULTS_FILE, CHECKPOINT_FILE)
results_df

No checkpoint found. Starting fresh.

RUNNING 2 EXPERIMENTS


EXPERIMENT 1/2 (ID: 0)
  sft_samples: 20
  dpo_pairs: 3
  candidates_per_question: 4
  sft_dataset_name: tatsu-lab/alpaca
  eval_split: test
  eval_max_samples: 20
  random_seed: 42
  base_model: Qwen/Qwen2.5-0.5B
  rm_base_model: Qwen/Qwen2.5-0.5B
  lora_r: 4
  lora_alpha: 8
  dpo_beta: 0.1
  gradient_accumulation_steps: 4
  quantization_bits: 0
  iterative_dpo_rounds: 1
  sft_epochs: 1
  rm_epochs: 1
  dpo_epochs: 1
  sft_batch_size: 1
  rm_batch_size: 1
  dpo_batch_size: 1
  temperature: 0.7
  max_new_tokens_gen: 64
  max_new_tokens_eval: 128
  eval_batch_size: 1
  use_rm_scoring: False
  rm_weight: 0.5
  correctness_weight: 1.0

EXPERIMENT 0

🎮 GPU Memory: 0.00 GB allocated, 0.00 GB reserved
🎲 Random seeds set to 42

[Stage 1/6] Cold-Start SFT…


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss


✅ SFT Complete. 'coldstart_dorado' saved.

DPO ROUND 1/1

[Stage 2/6] Candidate Generation (Round 1)…


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

KeyboardInterrupt: 